# Cross-dataset calibration transfer

완료된 calibration artifact를 고정한 뒤 `DATASET_IDS`로 선택한 target에 적용하는 protocol-aware runbook입니다. calibration source와 target 선택을 분리하며 canonical `research.experiments.cross_dataset_calibration` API만 호출합니다.

- 1:N maximum-gallery-score threshold는 동일 score/protocol/codec 계약의 1:N target에만 적용됩니다.
- RFW-Official은 1:1 pair-score TAR/FAR/EER이므로 1:N threshold와 호환되지 않습니다.
- strict external transfer는 서로 다른 물리 dataset과 검증된 identity disjoint audit가 필요합니다.
- RFW-Custom→RFW-Official 같은 동일 RFW population은 명시적 same-domain diagnostic으로만 허용됩니다.
- 각 `ProtocolScoreContract`는 `checkpoint_training_identity_overlap_status`를 필수로 기록합니다.
- EdgeFace–RFW overlap은 `UNKNOWN`; strict unseen-identity 근거로 사용하지 않습니다.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun 프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments.cross_dataset_calibration import (
    CalibrationThresholdArtifact,
    IdentityOverlapAudit,
    ProtocolScoreContract,
    build_cross_dataset_calibration_plan,
    calibration_evaluation_manifest,
    evaluate_external_calibration_transfer,
    evaluate_rfw_official_internal_baseline,
)
from research.runtime.hashing import sha256_file

# 사용자가 선택하는 source/target dataset. 논리 ID는 lfw, survface,
# rfw_custom(비공식 1:N), rfw(RFW-Official 1:1)를 사용합니다.
CALIBRATION_SOURCE_DATASET_IDS = tuple(
    globals().get("CALIBRATION_SOURCE_DATASET_IDS", ("lfw",))
)
DATASET_IDS = tuple(globals().get("DATASET_IDS", ("survface", "lfw")))

# 완료 artifact에서 구성한 canonical provenance 객체를 주입합니다.
CALIBRATION_SOURCES = tuple(globals().get("CALIBRATION_SOURCES", ()))
TARGET_SCORE_CONTRACTS = tuple(globals().get("TARGET_SCORE_CONTRACTS", ()))
IDENTITY_OVERLAP_AUDITS = tuple(globals().get("IDENTITY_OVERLAP_AUDITS", ()))
# key = target ProtocolScoreContract.score_artifact.artifact_uid, value = DataFrame
TARGET_SCORE_FRAMES = dict(globals().get("TARGET_SCORE_FRAMES", {}))

# 동일 physical dataset/source population을 명시적 diagnostic으로 볼 때만 True.
# 서로 다른 dataset의 UNKNOWN audit를 우회하는 옵션이 아닙니다.
ALLOW_SAME_DOMAIN_DIAGNOSTIC = bool(
    globals().get("ALLOW_SAME_DOMAIN_DIAGNOSTIC", False)
)
EXECUTE_TRANSFER = bool(globals().get("EXECUTE_TRANSFER", False))
WRITE_TRANSFER_OUTPUTS = bool(globals().get("WRITE_TRANSFER_OUTPUTS", False))
TRANSFER_OUTPUT_ROOT = PROJECT_ROOT / "results/paper/calibration_transfer"

# Optional RFW-Official internal 9-fold baseline:
# {"target": ProtocolScoreContract, "pairs": DataFrame, "scores": Series,
#  "thresholds": tuple[float, ...], "strict_official": bool}
RFW_INTERNAL_BASELINE_SPEC = globals().get("RFW_INTERNAL_BASELINE_SPEC", None)


## 1. Source calibration과 target matrix 고정

source artifact, target score contract, codec SHA, protocol UID, physical dataset, source population UID, pairwise identity-overlap audit와 checkpoint-training overlap status를 먼저 검증합니다. source와 target score statistic이 다르거나 strict external 조건이 충족되지 않으면 평가 전에 중단합니다.


In [ ]:
selected_sources = tuple(
    source
    for source in CALIBRATION_SOURCES
    if set(source.source_dataset_ids).issubset(
        set(CALIBRATION_SOURCE_DATASET_IDS)
    )
)
TRANSFER_PLAN = None
if selected_sources and TARGET_SCORE_CONTRACTS:
    TRANSFER_PLAN = build_cross_dataset_calibration_plan(
        calibration_sources=selected_sources,
        targets=TARGET_SCORE_CONTRACTS,
        target_dataset_ids=DATASET_IDS,
        identity_overlap_audits=IDENTITY_OVERLAP_AUDITS,
        allow_same_domain_diagnostic=ALLOW_SAME_DOMAIN_DIAGNOSTIC,
    )
    display(pd.DataFrame.from_records([
        {
            "case_uid": case.case_uid,
            "calibration_sources": ",".join(case.source.source_dataset_ids),
            "target": case.target.dataset_id,
            "protocol_kind": case.target.protocol_kind,
            "score_statistic": case.target.score_statistic,
            "transfer_scope": case.transfer_scope,
            "evaluation_mode": case.as_dict()["evaluation_mode"],
        }
        for case in TRANSFER_PLAN.cases
    ]))
else:
    print(
        "PREVIEW ONLY: CALIBRATION_SOURCES와 TARGET_SCORE_CONTRACTS를 "
        "완료 artifact에서 구성해 주입하십시오."
    )


## 2. Frozen threshold 적용 및 protocol-aware 평가

`EXECUTE_TRANSFER=True`일 때만 target score를 평가합니다. RFW-Official external fixed-threshold diagnostic과 RFW internal other-9-fold baseline은 서로 다른 evaluation mode와 artifact로 기록합니다.


In [ ]:
TRANSFER_EVALUATIONS = []
RFW_INTERNAL_EVALUATION = None

if EXECUTE_TRANSFER:
    if TRANSFER_PLAN is None:
        raise RuntimeError("transfer plan이 구성되지 않았습니다.")
    for case in TRANSFER_PLAN.cases:
        score_key = case.target.score_artifact.artifact_uid
        if score_key not in TARGET_SCORE_FRAMES:
            raise KeyError(f"target score frame이 없습니다: {score_key}")
        frame = TARGET_SCORE_FRAMES[score_key]
        if not isinstance(frame, pd.DataFrame):
            raise TypeError(f"TARGET_SCORE_FRAMES[{score_key!r}]는 DataFrame이어야 합니다.")
        evaluation = evaluate_external_calibration_transfer(case, frame)
        TRANSFER_EVALUATIONS.append(evaluation)

    if RFW_INTERNAL_BASELINE_SPEC is not None:
        spec = dict(RFW_INTERNAL_BASELINE_SPEC)
        RFW_INTERNAL_EVALUATION = evaluate_rfw_official_internal_baseline(
            spec["target"],
            spec["pairs"],
            scores=spec["scores"],
            thresholds=spec["thresholds"],
            strict_official=bool(spec.get("strict_official", True)),
            bootstrap_seed=int(spec.get("bootstrap_seed", 42)),
            bootstrap_repeats=int(spec.get("bootstrap_repeats", 2000)),
        )

evaluation_rows = [evaluation.summary for evaluation in TRANSFER_EVALUATIONS]
if RFW_INTERNAL_EVALUATION is not None:
    evaluation_rows.append(RFW_INTERNAL_EVALUATION.summary)
TRANSFER_SUMMARY = pd.DataFrame.from_records(evaluation_rows)
display(TRANSFER_SUMMARY)


## 3. Compact immutable outputs

각 evaluation UID별로 scored rows, group summary와 lineage manifest를 새 디렉터리에 기록합니다. 기존 디렉터리는 덮어쓰지 않습니다.


In [ ]:
WRITTEN_OUTPUTS = []
all_evaluations = list(TRANSFER_EVALUATIONS)
if RFW_INTERNAL_EVALUATION is not None:
    all_evaluations.append(RFW_INTERNAL_EVALUATION)

if WRITE_TRANSFER_OUTPUTS:
    if not EXECUTE_TRANSFER:
        raise RuntimeError("WRITE_TRANSFER_OUTPUTS=True에는 EXECUTE_TRANSFER=True가 필요합니다.")
    for evaluation in all_evaluations:
        destination = TRANSFER_OUTPUT_ROOT / evaluation.evaluation_uid
        destination.mkdir(parents=True, exist_ok=False)
        scored_path = destination / "scored_rows.csv"
        group_path = destination / "group_summary.csv"
        manifest_path = destination / "manifest.json"
        evaluation.scored_rows.to_csv(
            scored_path, index=False, encoding="utf-8", lineterminator="\n"
        )
        evaluation.group_summary.to_csv(
            group_path, index=False, encoding="utf-8", lineterminator="\n"
        )
        manifest = calibration_evaluation_manifest(evaluation)
        manifest["outputs"] = {
            "scored_rows.csv": {
                "bytes": scored_path.stat().st_size,
                "sha256": sha256_file(scored_path),
            },
            "group_summary.csv": {
                "bytes": group_path.stat().st_size,
                "sha256": sha256_file(group_path),
            },
        }
        manifest_path.write_text(
            json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        WRITTEN_OUTPUTS.append(str(destination))
WRITTEN_OUTPUTS


## 해석 경계

- `DATASET_IDS`는 target 선택이며 calibration source 선택과 독립입니다.
- 1:N FPIR calibration과 1:1 FAR calibration은 score statistic이 달라 서로 전이하지 않습니다.
- RFW-Official EER는 held-out pair score/label에서 산출하며 other-9-fold Accuracy/TAR/FAR threshold와 provenance를 분리합니다.
- `same_domain_overlap_diagnostic`은 cross-domain 또는 strict unseen-identity 결과가 아닙니다.
- EdgeFace–RFW overlap은 `UNKNOWN`; 모델 비교는 checkpoint-level로 한정합니다.
- 이 노트북은 Faiss IVF-PQ, pgvector IVFFlat, ANN sweep, BalancedFace, uncertainty/defer 실험을 실행하지 않습니다.
